In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
data_path = Path(os.environ["DATA_PATH"])
generated_path = data_path / "generated"
scripts_path = Path("./scripts")

In [60]:
df_init = (
    pd.read_excel(
        data_path / "processing" / "RPPC_vivienda-interes-social-nueva_2020-2024.xlsx",
    )
    .rename(
        columns={
            "Inmobiliaria": "inmobiliaria",
            "Lote": "lote",
            "Manzana": "manzana",
            "Fraccionamiento": "fraccionamiento",
            "Direccion": "direccion",
            "Razón social": "razon_social",
            "Comprador": "comprador",
            "Acreedor": "acreedor",
            "Acreedores": "acreedores",
            "Tipo de vivienda": "tipo_vivienda",
            "Categoría": "categoria",
            "Folio real": "folio",
            "Fecha de operacion": "fecha_operacion",
            "Competencia actual": "competencia_actual",
            "Valor de operacion": "valor_operacion",
            "Monto de crédito": "monto_credito",
            "Fecha de partida": "fecha_partida",
            "Partida": "partida",
            "Superficie": "mts_superficie",
            "Edad": "edad",
            "Latitud": "latitud",
            "Longitud": "longitud",
            "Mercado Exe": "mercado_exe",
        },
    )
    .assign(
        folio=lambda df: pd.to_numeric(df["folio"], errors="coerce").astype("Int64"),
        fecha_operacion=lambda df: pd.to_datetime(df["fecha_operacion"]),
        competencia_actual=lambda df: (
            df["competencia_actual"].str.lower().eq("sí").astype("Int64")
        ),
        valor_operacion=lambda df: pd.to_numeric(
            df["valor_operacion"],
            errors="coerce",
        ),
        monto_credito=lambda df: pd.to_numeric(df["monto_credito"], errors="coerce"),
        fecha_partida=lambda df: pd.to_datetime(df["fecha_partida"]),
        partida=lambda df: pd.to_numeric(df["partida"], errors="coerce").astype(
            "Int64",
        ),
        mts_superficie=lambda df: pd.to_numeric(df["mts_superficie"], errors="coerce"),
        edad=lambda df: pd.to_numeric(df["edad"], errors="coerce").astype("Int64"),
        latitud=lambda df: pd.to_numeric(df["latitud"], errors="coerce"),
        longitud=lambda df: pd.to_numeric(df["longitud"], errors="coerce"),
        mercado_exe=lambda df: df["mercado_exe"].str.lower().eq("sí"),
        direccion=lambda df: df["direccion"].astype(str).replace(np.nan, pd.NA),
        fraccionamiento=lambda df: df["fraccionamiento"],
    )
    .assign(
        fraccionamiento=lambda df: (
            df["fraccionamiento"]
            .str.strip()
            .replace(
                {
                    r"^FRACCIONAMIENTO VILLANOVA.*": "VILLANOVA",
                    r".*PORTICOS DEL VALLE.*": "PORTICOS DEL VALLE",
                },
                regex=True,
            )
            .where(
                lambda x: ~x.str.startswith("FRAC"),
                lambda x: (
                    x.str.replace("FRACCIONAMIENTO", "")
                    .str.replace("FRACC", "")
                    .str.replace("FRACCTO", "")
                    .str.replace("FRACTO", "")
                    .str.strip()
                ),
            )
            .where(
                lambda x: ~x.str.startswith("COLONIA BALBUENA"),
                lambda x: x.str.replace("DE MEXICALI BC DENTRO DE LA", "").str.strip(),
            )
            .where(
                lambda x: ~x.str.startswith("COLONIA GRANJAS AGRICOLAS"),
                lambda x: x.str.replace(
                    "FOLIO REAL 1598810 DE MEXICALI BC",
                    "",
                ).str.strip(),
            )
            .where(
                lambda x: ~x.str.startswith("LA RIOJA"),
                lambda x: x.str.replace("CASTILLAUNA", "CASTILLA").str.strip(),
            )
            .str.casefold()
            .replace(
                {
                    "gran foresta": "foresta residencial",
                    "foresta life": "foresta residencial",
                },
            )
        ),
        coords=lambda df: df["fraccionamiento"].map(
            {
                "angeles de puebla": (32.564770, -115.339935),
                "colonia balbuena": (32.630970, -115.472679),
                "corceles residencial": (32.567668, -115.469794),
                "gran foresta": (32.563446, -115.434414),
                "huertas del colorado": (32.563662, -115.411109),
                "la rioja seccion castillauna": (32.656198, -115.364974),
                "porticos del valle": (32.595718, -115.438633),
                "parajes de puebla": (32.557625, -115.346180),
                "quinta granada": (32.572234, -115.469416),
                "quinta granada 3": (32.567764, -115.473000),
                "san andres": (32.577850, -115.424811),
                "valle oriente": (32.571884, -115.361330),
                "privadas condesa": (32.595546, -115.344834),
            },
        ),
        new_lat=lambda df: df["coords"].apply(
            lambda x: x[0] if not pd.isna(x) else np.nan,
        ),
        new_lon=lambda df: df["coords"].apply(
            lambda x: x[1] if not pd.isna(x) else np.nan,
        ),
        latitud=lambda df: df["latitud"].fillna(df["new_lat"]),
        longitud=lambda df: df["longitud"].fillna(df["new_lon"]),
        privada=lambda df: (
            df["direccion"]
            .str.strip()
            .where(
                lambda x: ~x.str.startswith("DESARROLLO URBANO LA CONDESA"),
                lambda x: x.str.replace(
                    r"DESARROLLO URBANO LA CONDESA SECCI(O|Ó)N",
                    "WANTED_",
                    regex=True,
                ).str.strip(),
            )
            .where(
                lambda x: ~x.str.startswith("FRACCIONAMIENTO LA CONDESA SECCION"),
                lambda x: x.str.replace(
                    "FRACCIONAMIENTO LA CONDESA SECCION",
                    "WANTED_",
                ).str.strip(),
            )
            .where(
                lambda x: (
                    ~x.str.startswith(
                        "DESARROLLO URBANO VICTORIA RESIDENCIAL SEGUNDA SECCION",
                    )
                ),
                lambda x: x.str.replace(
                    "DESARROLLO URBANO VICTORIA RESIDENCIAL SEGUNDA SECCION",
                    "WANTED_VICTORIA SEGUNDA SECCION",
                ).str.strip(),
            )
            .where(lambda x: x.str.startswith("WANTED_"), None)
            .str.replace("WANTED_", "")
            .str.strip()
        ),
        direccion=lambda df: (
            df["direccion"].str.replace("LOCALIZACION:", "").str.strip()
        ),
    )
    .drop(columns=["coords", "new_lat", "new_lon"])
    .loc[lambda df: ~df["fraccionamiento"].str.match(r".*puerto de san felipe.*")]
    .loc[lambda df: df["fraccionamiento"] != "colonia granjas agricolas"]
    .dropna(subset=["fraccionamiento"])
    .reset_index(drop=True)
)

In [61]:
df_init.to_parquet(generated_path / "registro.parquet")

In [58]:
df_init["privada"].value_counts().sort_index()

privada
ATARES                        1
BAENA                       350
BAVIERA                       1
BENAVENTE                   606
FONTALBA                     13
GALVEZ                      362
GANTE                       505
GERONA                      442
LEGANÉS                     297
MAYORGA                     529
OLEAGA                      276
OLIVETO                     477
VELAYOS                     219
VICTORIA SEGUNDA SECCION      7
VILLAR                      379
Name: count, dtype: int64

In [30]:
df_init.loc[lambda df: df["privada"].str.contains("VICTORIA")]

,inmobiliaria,folio,fecha_operacion,Municipio,lote,manzana,fraccionamiento,direccion,razon_social,comprador,...,partida,mts_superficie,acreedores,tipo_vivienda,edad,latitud,longitud,mercado_exe,categoria,privada
